In [ ]:
# 1. Install zstd dependency first
!apt-get update && apt-get install -y zstd

# 2. Now run the Ollama installation script
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Install PyNgrok for creating your public tunnel
!pip install -q pyngrok

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,300 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,005 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubunt

In [ ]:
!nvidia-smi

Fri Jun  5 18:21:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# To pull the 3B model:
!ollama pull qwen2.5-coder:3b

# ALTERNATIVE (Recommended for even better answers):
# !ollama pull qwen2.5-coder:7b

In [ ]:
!ollama pull qwen2.5-coder:3b

In [ ]:
# 1. Clean up any stale processes from previous runs
print("🧹 Cleaning up old processes...")
!pkill -f ollama
!pkill -f cloudflared
!pkill -f ssh
!pkill -f lt

# 2. Install system dependencies (zstd for Ollama, Cloudflared for tunnel)
print("📦 Installing system dependencies...")
!apt-get update && apt-get install -y zstd > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 3. Install Ollama via official script
print("📥 Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time
import re

# 4. Configure Ollama to allow web connections and cross-origin requests
os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"

print("🚀 Starting Ollama Server...")
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

# Wait for server to initialize
time.sleep(5)

# 5. Pull your model
print("🤖 Pulling Qwen2.5-Coder 3B model (This may take a minute)...")
!ollama pull qwen2.5-coder:3b

# 6. Fire up Cloudflare Tunnel targeting Ollama's port (11434)
print("🌐 Creating public Cloudflare Tunnel...")
cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait and catch the URL from the logs
time.sleep(6)
cf_url = None

for _ in range(60):
    line = cf_process.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            cf_url = match.group(0)
            break

if cf_url:
    print("\n" + "="*60)
    print("🔗 COPY THIS EXACT URL FOR YOUR HTML FILE:")
    print(f"{cf_url}/api/chat")
    print("="*60)
else:
    print("\n❌ Tunnel setup timed out. Please run this cell one more time.")

🧹 Cleaning up old processes...
📦 Installing system dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,705 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,607 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu j